<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 2 · Tipos de dato que no engañan
**Excel Corporativo · 12 Claves** — Números, textos y fechas · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/02_tipos_de_dato.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-02)

**Al terminar podrás:**
1. Ver cuánto de una columna está guardado como texto.
2. Medir cuánto dinero deja por fuera una suma.
3. Entender por qué las fechas en varios formatos impiden analizar por mes.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · La exportación, como la ve Excel
La exportación del ERP llega con montos escritos con símbolo y puntos (`$ 1.234.567`), cantidades con la unidad pegada (`33 und`) y fechas en tres formatos. Para Excel, todo lo que no es claramente un número o una fecha queda como **texto**, y el texto no se suma ni se agrupa por mes.

In [ ]:
# @title Leer la hoja ERP_Sucio y ver qué tipo guarda Excel en cada celda
hoja = pd.read_excel(ruta("Distribuidora_Andina_practica.xlsx"), sheet_name="ERP_Sucio", dtype=object)
def tipo(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "Vacía"
    if isinstance(x, str): return "Texto (con unidad)" if "und" in x else "Texto"
    return "Número"
tipos = pd.DataFrame({c: hoja[c].map(tipo) for c in ["Valor Neto", "Cantidad", "Fecha"]})
conteo = pd.concat({c: tipos[c].value_counts() for c in tipos}, axis=1).fillna(0).astype(int)
display(conteo.style.format(miles))
display(Markdown("Así guarda Excel cada celda del libro de práctica. **Ojo:** si abres el CSV directamente, Excel intenta adivinar los tipos según la configuración regional del equipo, y el resultado cambia de un computador a otro. Por eso se practica con el libro."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué parte de cada columna llega como texto? |
> | **Qué se grafica** | Barras apiladas al 100 %: proporción de celdas que son número, texto o están vacías, por columna |
> | **Por qué esta forma** | Compara la composición de dos columnas de un vistazo. |
> | **Qué se descartó** | Una tabla de conteos: los números absolutos no muestran la proporción. |
> | **Cómo se lee** | El segmento magenta es lo que SUMA va a ignorar. |
> | **En Excel 365** | =CONTAR (números) y =CONTARA (no vacías) sobre cada columna. |

In [ ]:
# @title Gráfico 1 · Número, texto o vacía
prop = conteo / conteo.sum()
fig = go.Figure()
for estado, color in [("Número", AZUL), ("Texto", MAGENTA), ("Texto (con unidad)", MAGENTA), ("Vacía", GRIS)]:
    if estado in prop.index:
        fig.add_bar(name=estado, y=prop.columns, x=prop.loc[estado], orientation="h", marker_color=color,
                    text=[f"{v:.0%}" if v > 0.03 else "" for v in prop.loc[estado]], marker_line_color="white", marker_line_width=2)
txt_valor = prop.loc["Texto", "Valor Neto"]
fig.update_layout(barmode="stack", xaxis_tickformat=".0%", height=300, legend_title="",
                  title=f"Cantidad y Fecha llegaron completas como texto; Valor Neto, el {porc(txt_valor)}")
fig.show()

## 2 · ¿Cuánto dinero ignora SUMA?

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánto falta en el total si se suma la columna tal como llegó? |
> | **Qué se grafica** | Dos barras: lo que suma Excel (solo números) y el total real (después de convertir) |
> | **Por qué esta forma** | El mensaje es una diferencia entre dos cifras. |
> | **Qué se descartó** | Una tarjeta con el porcentaje: se pierde la magnitud en pesos. |
> | **Cómo se lee** | La distancia entre las barras es el dinero que el informe no mostraría. |
> | **En Excel 365** | Compara =SUMA(columna original) con =SUMA(columna convertida con VALOR). |

In [ ]:
# @title Gráfico 2 · Suma de Excel frente a total real
v = hoja["Valor Neto"]
es_txt = v.map(lambda x: isinstance(x, str))
suma_excel = v[~es_txt].dropna().astype(float).sum()
suma_texto = v[es_txt].str.replace(r"[$.\s]", "", regex=True).astype(float).sum()
valor_real = v.map(lambda x: float(x.replace("$", "").replace(".", "").strip()) if isinstance(x, str) else x).astype(float)
falta = suma_texto / (suma_excel + suma_texto)
comp = pd.Series({"Lo que suma Excel": suma_excel, "Total real": suma_excel + suma_texto}) / 1e6
fig = go.Figure(go.Bar(x=comp.values, y=comp.index, orientation="h", marker_color=[MAGENTA, AZUL],
                       text=[f"$ {v:,.0f} M".replace(",", ".") for v in comp.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"SUMA deja por fuera {millones(suma_texto)}: el {porc(falta)} del valor", xaxis_title=MM, height=260)
dar_aire(fig, comp.max())
fig.show()

## 3 · Fechas en tres formatos

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿En cuántos formatos llegaron las fechas? |
> | **Qué se grafica** | Barras: número de filas por formato de fecha |
> | **Por qué esta forma** | Pocas categorías, una medida. |
> | **Qué se descartó** | Torta: la pregunta no es la proporción sino que existan varios formatos. |
> | **Cómo se lee** | Cada barra es un formato que necesita su propia conversión. |
> | **En Excel 365** | Filtro de texto sobre la columna, o =ESNUMERO(FECHANUMERO(C2)). |

In [ ]:
# @title Gráfico 3 · Formatos de fecha
def formato_fecha(x):
    x = str(x)
    if "/" in x: return "dd/mm/aaaa"
    if "-" in x: return "aaaa-mm-dd"
    return "Número serial (como texto)"
fmt = hoja["Fecha"].map(formato_fecha).value_counts().sort_values()
fig = go.Figure(go.Bar(x=fmt.values, y=fmt.index, orientation="h", marker_color=[TEAL, TEAL, AZUL][-len(fmt):],
                       text=[miles(v) for v in fmt.values], textposition="outside", cliponaxis=False))
fig.update_layout(title=f"Las fechas llegaron en {len(fmt)} formatos distintos", xaxis_title="Filas", height=260)
dar_aire(fig, fmt.max())
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué se pierde si solo se convierte el formato más común? |
> | **Qué se grafica** | Líneas: venta mensual con las fechas convertidas según la opción elegida |
> | **Por qué esta forma** | Es una serie de tiempo: la línea muestra los meses que se pierden o quedan bajos. |
> | **Qué se descartó** | Una tabla mensual: los huecos no se ven. |
> | **Cómo se lee** | Compara las dos opciones: con una sola conversión, cada mes queda más bajo de lo real. |
> | **En Excel 365** | Columna con =SI.ERROR(FECHANUMERO(C2);VALOR(C2)) y luego tabla dinámica por mes. |

In [ ]:
# @title Gráfico 4 · Venta mensual según cómo se conviertan las fechas
conversion = "Todos los formatos"  # @param ["Solo dd/mm/aaaa", "Todos los formatos"]
def a_fecha(s):
    s = str(s)
    if "/" in s: return pd.to_datetime(s, format="%d/%m/%Y")
    if conversion == "Solo dd/mm/aaaa": return pd.NaT
    if "-" in s: return pd.to_datetime(s, format="%Y-%m-%d")
    return pd.Timestamp("1899-12-30") + pd.Timedelta(days=int(s))
x = hoja.assign(f=hoja["Fecha"].map(a_fecha), valor=valor_real)
perdidas = x["f"].isna().mean()
m = x.dropna(subset=["f"]).groupby(x["f"].dt.to_period("M"))["valor"].sum() / 1e6
fig = go.Figure(go.Scatter(x=m.index.to_timestamp(), y=m.values, mode="lines+markers", line=dict(color=AZUL if conversion != "Solo dd/mm/aaaa" else MAGENTA)))
eje_meses(fig, m.index.to_timestamp())
fig.update_layout(title=f"{conversion}: quedan por fuera el {porc(perdidas)} de las filas", yaxis_title=MM, height=340)
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · ¿Qué parte del valor de la columna Valor Neto ignora SUMA?
respuesta = "Elige una opción"  # @param ["Elige una opción", "10 %", "22 %", "35 %", "50 %"]
revisar(respuesta, f"{falta:.0%}".replace("%", " %"), """Es la distancia entre las dos barras del gráfico 2.""")

In [ ]:
# @title Reto 2 · Una celda de la columna Fecha muestra 45299. ¿Qué haces?
respuesta = "Elige una opción"  # @param ["Elige una opción", "Borrarla", "Convertirla con VALOR y darle formato de fecha", "Escribir la fecha a mano", "Nada: ya es una fecha"]
revisar(respuesta, "Convertirla con VALOR y darle formato de fecha", """45299 es un número serial que llegó como texto: VALOR lo vuelve número y el formato lo muestra como fecha.""")

In [ ]:
# @title Reto 3 · ¿Qué fórmula cuenta cuántos textos hay en una columna que debería ser numérica?
respuesta = "Elige una opción"  # @param ["Elige una opción", "=CONTAR(I:I)", "=CONTARA(I:I)-CONTAR(I:I)", "=SUMA(I:I)", "=CONTAR.BLANCO(I:I)"]
revisar(respuesta, "=CONTARA(I:I)-CONTAR(I:I)", """CONTARA cuenta lo no vacío; CONTAR, solo números. La diferencia son los textos.""")

## Apóyate en IA
Cuando una suma no cuadre, pide un diagnóstico del tipo de dato:

```text
Tengo una columna de Excel 365 donde =CONTAR da 1.450 y =CONTARA da 1.816.
Estos son 8 valores de muestra tal como aparecen: [pega aquí]
Dime qué tipos de problema hay y dame una fórmula en español (separador ;) para convertir todos en número.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* El formato cambia cómo se ve un valor, no lo que es.
* SUMA ignora los textos sin avisar: compara CONTAR con CONTARA.
* Una columna de fechas en varios formatos necesita una conversión por formato, o Power Query.

**Siguiente:** [Clave 3 · Referencias que se copian bien](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-03) · [Cuaderno 03](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/03_referencias.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>